# Module 3, Notebook 1: Stock-Watson Two-Step Estimator

## Learning Objectives

By completing this notebook, you will:

1. **Implement** the Stock-Watson two-step estimator from scratch
2. **Extract** latent factors via principal components analysis
3. **Estimate** factor dynamics using vector autoregression
4. **Compute** variance decomposition and R-squared statistics
5. **Validate** estimates against simulated ground truth
6. **Apply** the estimator to real macroeconomic data

## Prerequisites

- Module 1: Static factor models and PCA
- Module 2: VAR models and time series dynamics
- Linear algebra: eigenvalue decomposition
- Python: NumPy, Matplotlib

## Estimated Time: 90 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "the Stock-Watson two-step estimator from scratch",
    "latent factors via principal components analysis",
    "factor dynamics using vector autoregression",
    "variance decomposition and R-squared statistics",
    "estimates against simulated ground truth",
    "the estimator to real macroeconomic data"
])

## Setup and Imports

We begin by importing the necessary libraries and setting parameters for reproducibility.

In [ ]:
# Standard numerical and plotting libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.linalg import eigh, orthogonal_procrustes

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Display settings
pd.set_option('display.precision', 4)
np.set_printoptions(precision=4, suppress=True)

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Conceptual Foundation

### The Dynamic Factor Model

A Dynamic Factor Model (DFM) decomposes high-dimensional panel data $X$ (T observations × N variables) into:

**Measurement equation:**
$$X_{it} = \lambda_i' F_t + e_{it}$$

where:
- $F_t$ is an $r \times 1$ vector of latent factors (r << N)
- $\lambda_i$ is an $r \times 1$ vector of factor loadings for variable $i$
- $e_{it}$ is idiosyncratic error

**Dynamics equation:**
$$F_t = \Phi_1 F_{t-1} + \Phi_2 F_{t-2} + ... + \Phi_p F_{t-p} + \eta_t$$

The factors follow a VAR(p) process.

### The Stock-Watson Two-Step Approach

**Step 1: Factor Extraction via PCA**
- Compute sample covariance matrix: $\hat{\Sigma}_X = X'X / T$
- Eigenvalue decomposition: $\hat{\Sigma}_X = V D V'$
- Factor estimates: $\hat{F} = X V_r / \sqrt{T}$

**Step 2: Estimate Factor Dynamics via OLS**
- Regress $\hat{F}_t$ on $\hat{F}_{t-1}, ..., \hat{F}_{t-p}$
- Obtain $\hat{\Phi}_1, ..., \hat{\Phi}_p$ by least squares

**Key insight:** PCA consistently estimates the factor space under large N and T asymptotics, so we can treat estimated factors as if they were observed in Step 2.

## 2. Data Simulation

We simulate a DFM with known parameters to validate our implementation.

In [ ]:
def simulate_dynamic_factor_model(T=300, N=50, r=3, p=2, 
                                   noise_ratio=0.4, seed=42):
    """
    Simulate data from a dynamic factor model.
    
    Parameters
    ----------
    T : int
        Number of time periods
    N : int
        Number of variables
    r : int
        Number of factors
    p : int
        VAR lag order for factors
    noise_ratio : float
        Standard deviation of idiosyncratic noise
    seed : int
        Random seed
        
    Returns
    -------
    X : ndarray (T, N)
        Observed data
    F_true : ndarray (T, r)
        True factors
    Lambda_true : ndarray (N, r)
        True loadings
    Phi_true : ndarray (r, r, p)
        True VAR coefficients
    """
    np.random.seed(seed)
    
    # Step 1: Generate factor loadings with heterogeneous strengths
    Lambda_true = np.random.randn(N, r) * 0.8
    
    # Make first few loadings stronger (more factor-driven variables)
    Lambda_true[:10, :] *= 1.5
    
    # Step 2: Generate VAR(p) coefficients for factors
    Phi_true = np.zeros((r, r, p))
    
    # Lag 1: Dominant autoregressive component
    Phi_true[:, :, 0] = np.array([[0.7, 0.1, 0.05],
                                   [0.1, 0.6, 0.1],
                                   [0.05, 0.1, 0.65]])
    
    # Lag 2: Weaker second-order effects
    if p >= 2:
        Phi_true[:, :, 1] = np.random.randn(r, r) * 0.15
    
    # Step 3: Simulate factors with burn-in period
    burn_in = 100
    F_all = np.zeros((T + burn_in, r))
    
    # Initial values from stationary distribution
    F_all[:p] = np.random.randn(p, r) * 0.5
    
    # Generate factor dynamics
    for t in range(p, T + burn_in):
        # VAR evolution
        F_t = np.zeros(r)
        for lag in range(p):
            F_t += Phi_true[:, :, lag] @ F_all[t - lag - 1, :]
        
        # Add innovation
        F_t += np.random.randn(r) * 0.5
        F_all[t, :] = F_t
    
    # Remove burn-in
    F_true = F_all[-T:, :]
    
    # Step 4: Generate idiosyncratic errors with heteroscedasticity
    # Some variables have higher noise
    sigma_e = np.ones(N) * noise_ratio
    sigma_e[N//2:] *= 1.5  # Second half has more noise
    
    e = np.random.randn(T, N) * sigma_e
    
    # Step 5: Generate observed data
    X = F_true @ Lambda_true.T + e
    
    return X, F_true, Lambda_true, Phi_true


# Generate simulation data
print("Simulating dynamic factor model...\n")
X, F_true, Lambda_true, Phi_true = simulate_dynamic_factor_model(
    T=300, N=60, r=3, p=2, noise_ratio=0.4
)

T, N = X.shape
r_true = F_true.shape[1]
p_true = Phi_true.shape[2]

print(f"Dimensions:")
print(f"  T (time periods) = {T}")
print(f"  N (variables) = {N}")
print(f"  r (true factors) = {r_true}")
print(f"  p (VAR order) = {p_true}")
print(f"\nData summary:")
print(f"  Mean: {X.mean():.4f}")
print(f"  Std: {X.std():.4f}")
print(f"  Shape: {X.shape}")

The cell above generates synthetic data where we know the true factors and parameters. This allows us to validate our estimation procedure by comparing estimates to ground truth.

## 3. Step 1 - Factor Extraction via PCA

### Theoretical Background

Principal Components Analysis finds the directions of maximum variance in the data. For a factor model, these directions correspond to the factor space.

**Algorithm:**
1. Standardize data: $X_{std} = (X - \bar{X}) / \sigma_X$
2. Compute covariance: $\hat{\Sigma}_X = X_{std}' X_{std} / T$
3. Eigendecomposition: $\hat{\Sigma}_X V = V D$
4. Extract factors: $\hat{F} = X_{std} V_r / \sqrt{T}$

The normalization $1/\sqrt{T}$ ensures $\hat{F}'\hat{F}/T = I_r$.

In [ ]:
def extract_factors_pca(X, n_factors, standardize=True):
    """
    Extract factors using principal components analysis.
    
    Parameters
    ----------
    X : ndarray (T, N)
        Panel data
    n_factors : int
        Number of factors to extract
    standardize : bool
        Whether to standardize variables
        
    Returns
    -------
    F_hat : ndarray (T, r)
        Estimated factors
    Lambda_hat : ndarray (N, r)
        Estimated loadings
    eigenvalues : ndarray (N,)
        All eigenvalues (descending order)
    X_std : ndarray (T, N)
        Standardized data
    mean : ndarray (N,)
        Sample means
    std : ndarray (N,)
        Sample standard deviations
    """
    T, N = X.shape
    
    if n_factors > min(T, N):
        raise ValueError(f"n_factors must be <= min(T, N)")
    
    # Step 1: Standardize data
    mean = np.mean(X, axis=0)
    
    if standardize:
        std = np.std(X, axis=0, ddof=1)
        # Prevent division by zero
        std[std < 1e-10] = 1.0
        X_std = (X - mean) / std
    else:
        std = np.ones(N)
        X_std = X - mean
    
    # Step 2: Compute sample covariance matrix (N x N)
    Sigma_X = X_std.T @ X_std / T
    
    # Step 3: Eigenvalue decomposition
    # eigh returns eigenvalues in ascending order
    eigenvalues, eigenvectors = eigh(Sigma_X)
    
    # Reverse to descending order
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # Step 4: Extract top r eigenvectors
    V_r = eigenvectors[:, :n_factors]  # N x r
    
    # Step 5: Compute factor estimates
    # Normalization: F'F/T = I_r
    F_hat = X_std @ V_r / np.sqrt(T)  # T x r
    
    # Step 6: Compute loading estimates
    # Lambda = X' @ F / T
    Lambda_hat = X_std.T @ F_hat / T  # N x r
    
    return F_hat, Lambda_hat, eigenvalues, X_std, mean, std


# Extract factors from simulated data
print("Step 1: Extracting factors via PCA...\n")
F_hat, Lambda_hat, eigenvalues, X_std, mean, std = extract_factors_pca(
    X, n_factors=3, standardize=True
)

# Verify normalization
F_cov = F_hat.T @ F_hat / T
print("Factor covariance matrix (should be identity):")
print(F_cov)
print(f"\nIs F'F/T approximately I? {np.allclose(F_cov, np.eye(3), atol=1e-10)}")

# Display eigenvalues
print(f"\nTop 10 eigenvalues:")
print(eigenvalues[:10])
print(f"\nSum of top 3 eigenvalues: {eigenvalues[:3].sum():.4f}")
print(f"Total variance (trace): {eigenvalues.sum():.4f}")
print(f"Proportion explained: {eigenvalues[:3].sum() / eigenvalues.sum():.1%}")

### Visualization: Scree Plot

A scree plot shows how eigenvalues decay. We look for an "elbow" where additional factors explain little additional variance.

In [ ]:
# Purpose: Visualize eigenvalue decay to assess factor strength

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
n_show = 15
ax1.plot(range(1, n_show + 1), eigenvalues[:n_show], 'o-', 
         linewidth=2, markersize=8)
ax1.axvline(3, color='red', linestyle='--', linewidth=2, 
            label='True r=3', alpha=0.7)
ax1.set_xlabel('Factor Number', fontsize=12)
ax1.set_ylabel('Eigenvalue', fontsize=12)
ax1.set_title('Scree Plot', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cumulative variance explained
cumvar = np.cumsum(eigenvalues[:n_show]) / eigenvalues.sum()
ax2.plot(range(1, n_show + 1), cumvar, 'o-', 
         linewidth=2, markersize=8, color='green')
ax2.axhline(0.8, color='gray', linestyle=':', linewidth=2, 
            label='80% threshold')
ax2.axvline(3, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax2.set_xlabel('Number of Factors', fontsize=12)
ax2.set_ylabel('Cumulative Variance Explained', fontsize=12)
ax2.set_title('Cumulative Variance', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print(f"Three factors explain {cumvar[2]:.1%} of total variance")

## 4. Step 2 - Estimate Factor Dynamics

Now that we have factor estimates $\hat{F}_t$, we treat them as observed and estimate their dynamics using Vector Autoregression (VAR).

**VAR(p) model:**
$$\hat{F}_t = \Phi_1 \hat{F}_{t-1} + \Phi_2 \hat{F}_{t-2} + ... + \Phi_p \hat{F}_{t-p} + \eta_t$$

This is estimated by OLS regression.

In [ ]:
def estimate_factor_var(F, p_lags):
    """
    Estimate VAR(p) for factors using OLS.
    
    Parameters
    ----------
    F : ndarray (T, r)
        Factor estimates
    p_lags : int
        VAR lag order
        
    Returns
    -------
    Phi : ndarray (r, r, p)
        VAR coefficient matrices
    intercept : ndarray (r,)
        Intercept terms
    residuals : ndarray (T-p, r)
        VAR residuals
    residual_cov : ndarray (r, r)
        Covariance of residuals
    """
    T, r = F.shape
    
    if T <= p_lags:
        raise ValueError(f"Need T > p_lags, got T={T}, p={p_lags}")
    
    # Step 1: Construct regression matrices
    # Y = [F_{p+1}, F_{p+2}, ..., F_T]'
    Y = F[p_lags:, :]  # (T-p) x r
    
    # X = [1, F_t-1, F_t-2, ..., F_t-p] for each t
    X_lags = []
    for lag in range(1, p_lags + 1):
        X_lags.append(F[p_lags - lag : T - lag, :])
    
    X = np.column_stack([np.ones(T - p_lags)] + X_lags)  # (T-p) x (1 + r*p)
    
    # Step 2: OLS estimation
    # Minimize ||Y - X @ Beta||^2
    Beta, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
    
    # Step 3: Extract coefficients
    intercept = Beta[0, :]  # r x 1
    
    # Reshape Phi coefficients
    Phi = np.zeros((r, r, p_lags))
    for lag in range(p_lags):
        Phi[:, :, lag] = Beta[1 + lag*r : 1 + (lag+1)*r, :].T
    
    # Step 4: Compute residuals
    residuals = Y - X @ Beta
    residual_cov = np.cov(residuals, rowvar=False)
    
    return Phi, intercept, residuals, residual_cov


# Estimate VAR(2) for factors
print("Step 2: Estimating factor VAR(2) dynamics...\n")
Phi_hat, intercept, residuals, residual_cov = estimate_factor_var(F_hat, p_lags=2)

print("Estimated Phi_1 (lag 1 coefficient matrix):")
print(Phi_hat[:, :, 0])
print("\nTrue Phi_1:")
print(Phi_true[:, :, 0])

print("\nEstimated Phi_2 (lag 2 coefficient matrix):")
print(Phi_hat[:, :, 1])
print("\nTrue Phi_2:")
print(Phi_true[:, :, 1])

print("\nIntercept (should be near zero for demeaned factors):")
print(intercept)

print("\nResidual covariance matrix:")
print(residual_cov)

### Comparison with Ground Truth

Since PCA estimates factors only up to rotation, we align estimated factors to true factors using Procrustes analysis before comparing.

In [ ]:
# Purpose: Align estimated factors to true factors and compute correlation

# Procrustes alignment: Find rotation H that minimizes ||F_true - F_hat @ H||^2
H, _ = orthogonal_procrustes(F_hat, F_true)
F_aligned = F_hat @ H

# Compute correlation for each factor
correlations = np.array([
    np.corrcoef(F_true[:, i], F_aligned[:, i])[0, 1]
    for i in range(r_true)
])

print("Factor alignment quality (correlation after Procrustes):")
for i, corr in enumerate(correlations):
    print(f"  Factor {i+1}: {corr:.4f}")

print(f"\nMean absolute correlation: {np.abs(correlations).mean():.4f}")

High correlations (above 0.95) indicate excellent factor recovery. The Stock-Watson estimator successfully identifies the latent factors!

## 5. Complete Stock-Watson Estimator Class

Let's package our implementation into a reusable class with additional methods.

In [ ]:
class StockWatsonDFM:
    """
    Stock-Watson two-step estimator for Dynamic Factor Models.
    
    Parameters
    ----------
    n_factors : int
        Number of factors
    factor_lags : int, default=1
        VAR lag order for factors
    standardize : bool, default=True
        Standardize variables before PCA
    """
    
    def __init__(self, n_factors, factor_lags=1, standardize=True):
        self.n_factors = n_factors
        self.factor_lags = factor_lags
        self.standardize = standardize
        
        # Fitted parameters
        self.F_hat_ = None
        self.Lambda_hat_ = None
        self.Phi_hat_ = None
        self.eigenvalues_ = None
        self.mean_ = None
        self.std_ = None
        
    def fit(self, X):
        """Fit the model to data."""
        # Step 1: PCA
        results = extract_factors_pca(X, self.n_factors, self.standardize)
        self.F_hat_, self.Lambda_hat_, self.eigenvalues_, _, self.mean_, self.std_ = results
        
        # Step 2: VAR
        self.Phi_hat_, self.intercept_, self.residuals_, self.residual_cov_ = \
            estimate_factor_var(self.F_hat_, self.factor_lags)
        
        return self
    
    def variance_decomposition(self):
        """Compute variance explained by each factor."""
        if self.eigenvalues_ is None:
            raise ValueError("Model not fitted")
        
        total_var = self.eigenvalues_.sum()
        ratios = self.eigenvalues_[:self.n_factors] / total_var
        return ratios
    
    def variable_r_squared(self):
        """Compute R-squared for each variable."""
        if self.Lambda_hat_ is None:
            raise ValueError("Model not fitted")
        
        # R^2 = ||Lambda_i||^2 for standardized data
        return np.sum(self.Lambda_hat_**2, axis=1)
    
    def forecast(self, h=1):
        """Forecast factors h periods ahead."""
        if self.F_hat_ is None:
            raise ValueError("Model not fitted")
        
        T, r = self.F_hat_.shape
        p = self.factor_lags
        
        # Initialize forecast
        F_forecast = np.zeros((h, r))
        
        # Use last p observations as history
        F_history = np.vstack([self.F_hat_[-p:, :], F_forecast])
        
        for t in range(h):
            F_t = self.intercept_.copy()
            for lag in range(p):
                F_t += self.Phi_hat_[:, :, lag] @ F_history[p + t - lag - 1, :]
            F_forecast[t, :] = F_t
            F_history[p + t, :] = F_t
        
        return F_forecast


# Test the class
model = StockWatsonDFM(n_factors=3, factor_lags=2, standardize=True)
model.fit(X)

print("Model fitted successfully!")
print(f"\nVariance explained by factors:")
var_explained = model.variance_decomposition()
for i, ratio in enumerate(var_explained):
    print(f"  Factor {i+1}: {ratio:.1%}")
print(f"  Total: {var_explained.sum():.1%}")

print(f"\nAverage R-squared: {model.variable_r_squared().mean():.1%}")

## 6. Visualization: Factors and Loadings

Visual inspection helps interpret estimated factors and identify which variables load strongly on each factor.

In [ ]:
# Purpose: Plot estimated vs true factors and examine loadings

fig = plt.figure(figsize=(16, 10))

# Align factors for plotting
H, _ = orthogonal_procrustes(model.F_hat_, F_true)
F_aligned = model.F_hat_ @ H

# Plot 1-3: Factor time series
for i in range(3):
    ax = plt.subplot(3, 2, i*2 + 1)
    ax.plot(F_true[:, i], label='True', alpha=0.7, linewidth=2)
    ax.plot(F_aligned[:, i], label='Estimated', alpha=0.7, linewidth=2, linestyle='--')
    ax.set_title(f'Factor {i+1} Time Series', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time')
    ax.set_ylabel('Factor Value')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Plot 4-6: Loading distributions
for i in range(3):
    ax = plt.subplot(3, 2, i*2 + 2)
    ax.bar(range(N), model.Lambda_hat_[:, i], alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Factor {i+1} Loadings', fontsize=12, fontweight='bold')
    ax.set_xlabel('Variable')
    ax.set_ylabel('Loading')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Exercise 6.1: Implement Variable R-squared Analysis

Compute and visualize which variables are most strongly driven by the common factors vs idiosyncratic noise.

**Task:** Complete the function below to compute per-variable R-squared and create a visualization.

In [ ]:
# YOUR CODE HERE
# ---------------

def analyze_variable_r_squared(model, threshold=0.5):
    """
    Analyze which variables are most factor-driven.
    
    Parameters
    ----------
    model : StockWatsonDFM
        Fitted model
    threshold : float
        R-squared threshold for "high factor loading"
        
    Returns
    -------
    r_squared : ndarray (N,)
        R-squared for each variable
    high_loading_vars : ndarray
        Indices of variables with R^2 > threshold
    """
    # TODO: Compute R-squared for each variable
    # Hint: R^2_i = sum_j lambda_ij^2 for standardized data
    r_squared = None  # Replace this
    
    # TODO: Find variables exceeding threshold
    high_loading_vars = None  # Replace this
    
    # TODO: Create visualization
    # - Bar plot of R-squared values
    # - Horizontal line at threshold
    # - Highlight high-loading variables
    
    return r_squared, high_loading_vars


# Test your implementation
# r_squared, high_vars = analyze_variable_r_squared(model, threshold=0.5)

<details>
<summary>Hint 1</summary>
For standardized data, the R-squared is simply the sum of squared loadings: $R_i^2 = \sum_{j=1}^r \lambda_{ij}^2$. Use model.Lambda_hat_ to access loadings.
</details>

<details>
<summary>Hint 2</summary>
Use np.where(r_squared > threshold)[0] to find indices of high-loading variables.
</details>

In [ ]:
# AUTO-GRADED TESTS
# ------------------

def test_exercise_6_1():
    """Test variable R-squared analysis."""
    r_squared, high_vars = analyze_variable_r_squared(model, threshold=0.5)
    
    # Test 1: Correct shape
    assert r_squared is not None, "Don't forget to compute r_squared!"
    assert r_squared.shape == (N,), f"r_squared should have shape ({N},), got {r_squared.shape}"
    
    # Test 2: Valid range
    assert np.all((r_squared >= 0) & (r_squared <= 1)), "R-squared must be in [0, 1]"
    
    # Test 3: High-loading variables identified
    assert high_vars is not None, "Don't forget to identify high_loading_vars!"
    assert len(high_vars) > 0, "Should identify at least some high-loading variables"
    
    # Test 4: Threshold correctly applied
    assert np.all(r_squared[high_vars] > 0.5), "All high_vars should have R^2 > threshold"
    
    print("✅ Exercise 6.1 passed!")
    print(f"\nIdentified {len(high_vars)} high-loading variables (R^2 > 0.5)")
    print(f"Mean R-squared: {r_squared.mean():.3f}")
    print(f"Max R-squared: {r_squared.max():.3f}")

# Uncomment to test:
# test_exercise_6_1()

### Solution to Exercise 6.1

In [ ]:
# SOLUTION
# --------

def analyze_variable_r_squared(model, threshold=0.5):
    """
    Analyze which variables are most factor-driven.
    """
    # Compute R-squared: sum of squared loadings
    r_squared = np.sum(model.Lambda_hat_**2, axis=1)
    
    # Identify high-loading variables
    high_loading_vars = np.where(r_squared > threshold)[0]
    
    # Visualization
    fig, ax = plt.subplots(figsize=(14, 5))
    
    colors = ['red' if r > threshold else 'steelblue' for r in r_squared]
    bars = ax.bar(range(len(r_squared)), r_squared, color=colors, alpha=0.7)
    
    ax.axhline(threshold, color='black', linestyle='--', linewidth=2,
               label=f'Threshold = {threshold}')
    ax.set_xlabel('Variable Index', fontsize=12)
    ax.set_ylabel('R-squared', fontsize=12)
    ax.set_title('Factor Model Fit by Variable', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    return r_squared, high_loading_vars

# Run solution
r_squared, high_vars = analyze_variable_r_squared(model, threshold=0.5)
print(f"\n{len(high_vars)} variables have R^2 > 0.5")
print(f"These are highly driven by common factors.")

## Exercise 6.2: Factor Forecasting

Use the estimated VAR to forecast factors 10 periods ahead and visualize forecast uncertainty.

**Task:** Implement multi-step-ahead forecasting with confidence intervals.

In [ ]:
# YOUR CODE HERE
# ---------------

def forecast_factors_with_ci(model, h=10, alpha=0.05):
    """
    Forecast factors with confidence intervals.
    
    Parameters
    ----------
    model : StockWatsonDFM
        Fitted model
    h : int
        Forecast horizon
    alpha : float
        Significance level (e.g., 0.05 for 95% CI)
        
    Returns
    -------
    F_forecast : ndarray (h, r)
        Point forecasts
    F_lower : ndarray (h, r)
        Lower confidence bounds
    F_upper : ndarray (h, r)
        Upper confidence bounds
    """
    # TODO: Obtain point forecasts from model
    F_forecast = None  # Replace this
    
    # TODO: Compute forecast standard errors
    # Hint: Variance accumulates over forecast horizon
    # Use model.residual_cov_ for innovation variance
    
    # TODO: Construct confidence intervals
    # Use normal critical value: stats.norm.ppf(1 - alpha/2)
    F_lower = None  # Replace this
    F_upper = None  # Replace this
    
    return F_forecast, F_lower, F_upper


# Test your implementation
# F_fc, F_lo, F_hi = forecast_factors_with_ci(model, h=10, alpha=0.05)

<details>
<summary>Hint 1</summary>
Use model.forecast(h) to get point forecasts.
</details>

<details>
<summary>Hint 2</summary>
For a VAR(p), the h-step-ahead forecast variance is approximately h * residual_cov for large h. For small h, it accumulates as var(h) = Q + Phi @ var(h-1) @ Phi'.
</details>

In [ ]:
# AUTO-GRADED TESTS
# ------------------

def test_exercise_6_2():
    """Test factor forecasting."""
    h = 10
    F_fc, F_lo, F_hi = forecast_factors_with_ci(model, h=h, alpha=0.05)
    
    # Test 1: Correct shapes
    assert F_fc is not None, "Don't forget to compute forecasts!"
    assert F_fc.shape == (h, 3), f"Forecast should have shape ({h}, 3)"
    assert F_lo.shape == (h, 3), f"Lower bound should have shape ({h}, 3)"
    assert F_hi.shape == (h, 3), f"Upper bound should have shape ({h}, 3)"
    
    # Test 2: Bounds are properly ordered
    assert np.all(F_lo <= F_fc), "Lower bound should be <= point forecast"
    assert np.all(F_fc <= F_hi), "Point forecast should be <= upper bound"
    
    # Test 3: Reasonable uncertainty
    widths = F_hi - F_lo
    assert np.all(widths > 0), "Confidence intervals should have positive width"
    
    print("✅ Exercise 6.2 passed!")
    print(f"\nAverage 95% CI width: {widths.mean():.3f}")
    print(f"CI width increases with horizon: {widths[-1].mean() > widths[0].mean()}")

# Uncomment to test:
# test_exercise_6_2()

### Solution to Exercise 6.2

In [ ]:
# SOLUTION
# --------

def forecast_factors_with_ci(model, h=10, alpha=0.05):
    """
    Forecast factors with confidence intervals.
    """
    # Point forecasts
    F_forecast = model.forecast(h=h)
    
    # Compute forecast standard errors
    # For simplicity, use approximation: variance grows linearly
    # More accurate: recursively compute MSE matrices
    sigma_innov = np.sqrt(np.diag(model.residual_cov_))
    
    # Approximate h-step-ahead standard error
    # This is simplified; full computation requires MSE matrices
    se_forecast = np.zeros((h, model.n_factors))
    for i in range(h):
        se_forecast[i, :] = sigma_innov * np.sqrt(i + 1)
    
    # Critical value for normal distribution
    z_crit = stats.norm.ppf(1 - alpha / 2)
    
    # Construct confidence intervals
    F_lower = F_forecast - z_crit * se_forecast
    F_upper = F_forecast + z_crit * se_forecast
    
    # Visualization
    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    
    for i in range(3):
        ax = axes[i]
        
        # Historical data
        T = len(model.F_hat_)
        ax.plot(range(T), model.F_hat_[:, i], label='Historical', 
                linewidth=2, alpha=0.7)
        
        # Forecast
        forecast_idx = range(T, T + h)
        ax.plot(forecast_idx, F_forecast[:, i], label='Forecast',
                linewidth=2, color='red', alpha=0.7)
        
        # Confidence interval
        ax.fill_between(forecast_idx, F_lower[:, i], F_upper[:, i],
                        alpha=0.3, color='red', label='95% CI')
        
        ax.axvline(T, color='black', linestyle='--', linewidth=1, alpha=0.5)
        ax.set_title(f'Factor {i+1} Forecast', fontsize=12, fontweight='bold')
        ax.set_xlabel('Time')
        ax.set_ylabel('Factor Value')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return F_forecast, F_lower, F_upper

# Run solution
F_fc, F_lo, F_hi = forecast_factors_with_ci(model, h=10, alpha=0.05)
print("\nForecasts computed with 95% confidence intervals")

## Summary

### Key Takeaways

1. **Two-step estimation is fast and robust:** PCA in Step 1 + OLS in Step 2 avoids iterative optimization

2. **Standardization is crucial:** Variables must be on comparable scales for PCA to work correctly

3. **Factors are identified up to rotation:** Use Procrustes alignment when comparing to ground truth

4. **Asymptotic justification:** Consistency requires both N and T large, with N growing faster than T

5. **Variance decomposition:** R-squared statistics reveal which variables are factor-driven vs idiosyncratic

6. **Factor forecasting:** VAR dynamics enable multi-step-ahead predictions with confidence intervals

### What's Next

In the next notebook, we'll tackle the crucial question: **How many factors should we extract?** We'll implement Bai-Ng information criteria and visual diagnostics for data-driven factor number selection.

### Additional Resources

- Stock & Watson (2002): "Forecasting Using Principal Components from a Large Number of Predictors"
- Bai (2003): "Inferential Theory for Factor Models of Large Dimensions" 
- Module guide: `guides/01_stock_watson_estimator.md`

---

**Completion time:** You've completed Module 3, Notebook 1. Estimated time remaining: 60 minutes for Notebook 2.